# 1 · prepare_data — run once  *(v2: aligned with run5's unified class space)*
Builds everything the training grid needs:
1. **PKU label remap** (only if your local PKU export order differs from the unified space).
2. **ModRand datasets** — modality-randomized copies of PKU / Kaggle / DeepPCB train sets (photometric only → labels copied unchanged). Zero-shot safe.
3. **DeepPCB train/val split** — carves a seeded 90/10 split out of `deeppcb_yolo/trainval` (already unified by run3/run5; **no reconversion needed**).
4. **data.yaml files** for every grid row, written to `experiments/yamls/`.

Edit the **CONFIG** cell, then Run All.

In [ ]:
# --- project root resolution (same pattern as run1-run5) ---
import os, sys, json
from pathlib import Path

def resolve_project_root() -> Path:
    env = os.environ.get("PCB_PROJECT_ROOT")
    if env:
        return Path(env).resolve()
    cwd = Path.cwd().resolve()
    if cwd.name == "experiments":
        return cwd.parent
    if (cwd / "experiments").is_dir():
        return cwd
    return cwd

PROJECT_ROOT = resolve_project_root()
EXP_DIR = PROJECT_ROOT / "experiments"
sys.path.insert(0, str(EXP_DIR))   # mr_yolo11.py + make_modrand_dataset.py live here
print("PROJECT_ROOT:", PROJECT_ROOT)

In [ ]:
# ====================== CONFIG — EDIT THESE ======================
DATA     = PROJECT_ROOT / "datasets"
GEN      = DATA / "generated"          # all outputs of this notebook
YAML_DIR = EXP_DIR / "yamls"

# ---- unified class space == Run2/Kaggle/joint order (run5's NAMES) ----
CANONICAL_NAMES = ['mouse_bite', 'spur', 'missing_hole', 'short',
                   'open_circuit', 'spurious_copper']

# PKU-483 (Run1 source), YOLO layout: {split}/images + {split}/labels
PKU_TRAIN_IMG = DATA / "pku/train/images"; PKU_TRAIN_LBL = DATA / "pku/train/labels"
PKU_VAL_IMG   = DATA / "pku/val/images";   PKU_VAL_LBL   = DATA / "pku/val/labels"
PKU_TEST_IMG  = DATA / "pku/test/images";  PKU_TEST_LBL  = DATA / "pku/test/labels"

# Class order of YOUR local PKU labels — copy the `names` list from Run1's data.yaml.
# If it differs from CANONICAL_NAMES, the remap cell below rewrites the labels.
PKU_CURRENT = ['missing_hole', 'mouse_bite', 'open_circuit',
               'short', 'spur', 'spurious_copper']           # <-- EDIT to match Run1

# Kaggle (run5 already asserted its order == unified; kagglehub cache path works too)
KAG_TRAIN_IMG = DATA / "kaggle/train/images"; KAG_TRAIN_LBL = DATA / "kaggle/train/labels"
KAG_VAL_IMG   = DATA / "kaggle/valid/images"; KAG_TEST_IMG  = DATA / "kaggle/test/images"

# DeepPCB: ALREADY converted + unified by run3/run5 -> deeppcb_yolo/{trainval,test}
DPCB_ROOT = PROJECT_ROOT / "deeppcb_yolo"

PKU_K, KAGGLE_K, DPCB_K = 3, 1, 2      # ModRand variants per image

In [ ]:
# imports + sanity counts
import cv2, random, shutil
from make_modrand_dataset import random_variant, IMG_EXTS
random.seed(0)

def count(p):
    p = Path(p)
    return sum(1 for f in p.iterdir() if f.suffix.lower() in IMG_EXTS) if p.exists() else -1

for name, p in [("PKU train", PKU_TRAIN_IMG), ("PKU val", PKU_VAL_IMG), ("PKU test", PKU_TEST_IMG),
                ("Kaggle train", KAG_TRAIN_IMG), ("Kaggle val", KAG_VAL_IMG),
                ("DPCB trainval", DPCB_ROOT / "trainval/images"),
                ("DPCB test", DPCB_ROOT / "test/images")]:
    print(f"{name:14s} {count(p):6d}   {p}")
# -1 means the path is wrong -> fix CONFIG before continuing

In [ ]:
# PKU -> unified space (no-op if PKU_CURRENT already == CANONICAL_NAMES)
def remap_label_tree(splits, cur_names, dst_root):
    m = {cur_names.index(n): CANONICAL_NAMES.index(n) for n in cur_names}
    out = {}
    for split, (si, sl) in splits.items():
        di, dl = dst_root / split / "images", dst_root / split / "labels"
        di.mkdir(parents=True, exist_ok=True)
        dl.mkdir(parents=True, exist_ok=True)
        for f in Path(si).iterdir():
            if f.suffix.lower() in IMG_EXTS:
                shutil.copy2(f, di / f.name)
        for f in Path(sl).glob("*.txt"):
            rows = []
            for ln in f.read_text().splitlines():
                p = ln.split()
                if p:
                    p[0] = str(m[int(p[0])])
                    rows.append(" ".join(p))
            (dl / f.name).write_text("\n".join(rows))
        out[split] = (di, dl)
    return out

if PKU_CURRENT != CANONICAL_NAMES:
    new = remap_label_tree({"train": (PKU_TRAIN_IMG, PKU_TRAIN_LBL),
                            "val":   (PKU_VAL_IMG,   PKU_VAL_LBL),
                            "test":  (PKU_TEST_IMG,  PKU_TEST_LBL)},
                           PKU_CURRENT, GEN / "pku_canon")
    (PKU_TRAIN_IMG, PKU_TRAIN_LBL) = new["train"]
    (PKU_VAL_IMG,   PKU_VAL_LBL)   = new["val"]
    (PKU_TEST_IMG,  PKU_TEST_LBL)  = new["test"]
    print("PKU labels remapped into", GEN / "pku_canon")
else:
    print("PKU order already unified — no remap needed")

In [ ]:
# ModRand generator (photometric only -> labels copied verbatim)
def generate_modrand(src_img, src_lbl, out_root, k):
    out_img, out_lbl = Path(out_root) / "images", Path(out_root) / "labels"
    out_img.mkdir(parents=True, exist_ok=True)
    out_lbl.mkdir(parents=True, exist_ok=True)
    pool = sorted(p for p in Path(src_img).iterdir() if p.suffix.lower() in IMG_EXTS)
    for p in pool:
        img = cv2.imread(str(p))
        if img is None:
            continue
        lbl = Path(src_lbl) / (p.stem + ".txt")
        shutil.copy2(p, out_img / p.name)
        if lbl.exists():
            shutil.copy2(lbl, out_lbl / lbl.name)
        for i in range(k):
            cv2.imwrite(str(out_img / f"{p.stem}__mr{i}.jpg"), random_variant(img, pool))
            if lbl.exists():
                shutil.copy2(lbl, out_lbl / f"{p.stem}__mr{i}.txt")
    print(f"{out_root}: {sum(1 for _ in out_img.iterdir())} images")

generate_modrand(PKU_TRAIN_IMG, PKU_TRAIN_LBL, GEN / "pku_modrand_train", PKU_K)

In [ ]:
# Kaggle ModRand (8.5k images, k=1 -> ~17k; several minutes)
generate_modrand(KAG_TRAIN_IMG, KAG_TRAIN_LBL, GEN / "kaggle_modrand_train", KAGGLE_K)

In [ ]:
# DeepPCB: carve seeded 90/10 train/val out of trainval (labels already unified)
src_i, src_l = DPCB_ROOT / "trainval/images", DPCB_ROOT / "trainval/labels"
if (DPCB_ROOT / "train/images").exists():
    print("DeepPCB train/val already present — skipping")
else:
    imgs = sorted(p for p in src_i.iterdir() if p.suffix.lower() in IMG_EXTS)
    random.Random(42).shuffle(imgs)
    n_val = round(len(imgs) * 0.1)
    for split, items in [("val", imgs[:n_val]), ("train", imgs[n_val:])]:
        di, dl = DPCB_ROOT / split / "images", DPCB_ROOT / split / "labels"
        di.mkdir(parents=True, exist_ok=True)
        dl.mkdir(parents=True, exist_ok=True)
        for p in items:
            shutil.copy2(p, di / p.name)
            lp = src_l / (p.stem + ".txt")
            if lp.exists():
                shutil.copy2(lp, dl / lp.name)
        print(f"deeppcb {split}: {len(items)} images")

In [ ]:
# DeepPCB ModRand for row I (invert + FDA are the useful ops on binary images)
generate_modrand(DPCB_ROOT / "train/images", DPCB_ROOT / "train/labels",
                 GEN / "deeppcb_modrand_train", DPCB_K)

In [ ]:
# write all data.yaml files (unified names everywhere)
import yaml
YAML_DIR.mkdir(parents=True, exist_ok=True)

def write_yaml(name, train, val, test):
    d = {"path": str(PROJECT_ROOT),
         "train": str(Path(train).resolve()),
         "val": str(Path(val).resolve()),
         "test": str(Path(test).resolve()),
         "names": {i: n for i, n in enumerate(CANONICAL_NAMES)}}
    p = YAML_DIR / f"{name}.yaml"
    p.write_text(yaml.safe_dump(d, sort_keys=False))
    print("wrote", p)

write_yaml("pku",            PKU_TRAIN_IMG,                       PKU_VAL_IMG, PKU_TEST_IMG)
write_yaml("pku_modrand",    GEN / "pku_modrand_train/images",    PKU_VAL_IMG, PKU_TEST_IMG)
write_yaml("kaggle",         KAG_TRAIN_IMG,                       KAG_VAL_IMG, KAG_TEST_IMG)
write_yaml("kaggle_modrand", GEN / "kaggle_modrand_train/images", KAG_VAL_IMG, KAG_TEST_IMG)
write_yaml("deeppcb",         DPCB_ROOT / "train/images",          DPCB_ROOT / "val/images", DPCB_ROOT / "test/images")
write_yaml("deeppcb_modrand", GEN / "deeppcb_modrand_train/images", DPCB_ROOT / "val/images", DPCB_ROOT / "test/images")

In [ ]:
# visual sanity: 6 random ModRand variants — Otsu/adaptive ones should look DeepPCB-like
import matplotlib.pyplot as plt
mr = sorted((GEN / "pku_modrand_train/images").glob("*__mr*.jpg"))
sample = random.sample(mr, min(6, len(mr)))
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for ax, p in zip(axes.flat, sample):
    ax.imshow(cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB))
    ax.set_title(p.name, fontsize=7)
    ax.axis("off")
plt.tight_layout()
plt.show()